# 04 — Greedy Residue Crop Investigation

## Motivation

Global CA superposition (notebook 03) is a single rigid-body alignment. For large
proteins, even a good global RMSD hides local regions where the alignment is poor —
error propagates with distance from the alignment core.

The solution is **local crops**: define a canonical set of spatial patches on a
reference structure (one per cluster), then align each crop independently across
cluster members. Water conservation is then scored per-crop rather than globally.

## Algorithm

Greedy sphere-covering on residue canonical centers (CA for polymers, O for waters,
stored in `residues['atom_center']`):

1. Build a KD-tree on all present residue centers.
2. Pick the first uncovered residue as the seed.
3. All residues within `radius` Å of the seed form a crop; mark them covered.
4. Warn if any crop contains zero non-solvent residues (isolated waters).
5. Repeat until all residues are covered.

Crops may overlap (a residue can appear in more than one crop if multiple seed
spheres reach it). Once a residue is covered it cannot seed a new crop.

## This notebook

Demonstrates the crop on one representative structure from a small cluster,
reports crop statistics, and saves each crop as an mmCIF file for visual inspection.

## Setup

In [13]:
%load_ext autoreload
%autoreload 2


from cluster_filter import load_manifest_index, filter_clusters
from crop import count_nonsolvent_residues, get_residue_centers, greedy_crop, save_crop_as_mmcif
from boltzgen.data.data import Structure

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import sys
sys.path.insert(0, '../scripts')
sys.path.insert(0, '/data/rbg/users/gloriama/dev/foldeverything/src')

from pathlib import Path
import numpy as np

from cluster_filter import load_manifest_index, filter_clusters
from crop import count_nonsolvent_residues, get_residue_centers, greedy_crop, save_crop_as_mmcif
from boltzgen.data.data import Structure

CLUSTER_FILE   = '../clusters-by-entity-100.txt'
MANIFEST_FILE  = '/data/rbg/shared/datasets/processed_rcsb/rcsb_solvents/manifest.json'
STRUCTURES_DIR = Path('/data/rbg/shared/datasets/processed_rcsb/rcsb_solvents/structures')
OUTPUT_DIR     = Path('/data/rbg/users/gloriama/dev/water_conservation/outputs/crops')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Section 1 — Pick a representative structure

Load 2.0 Å filtered clusters and take the first small cluster (5–8 members).
Within that cluster, pick the member with the most non-solvent (polymer) residues
as the reference structure for cropping.

In [3]:
manifest_index = load_manifest_index(MANIFEST_FILE)
filtered, stats = filter_clusters(
    manifest_index,
    resolution_cutoff=2.0,
    min_cluster_size=5,
    cluster_file=CLUSTER_FILE,
)

# First cluster with 5–8 members
small_clusters = [c for c in filtered if 5 <= len(c) <= 8]
cluster = small_clusters[0]
print(f'Cluster ({len(cluster)} members): {cluster}')

Cluster (5 members): ['5kdo', '6crk', '6rmv', '8qeg', '8qeh']


In [4]:
# Pick the member with the most non-solvent residues
counts = {}
for pdb in cluster:
    npz_path = STRUCTURES_DIR / f'{pdb}.npz'
    npz = np.load(npz_path, allow_pickle=True)
    counts[pdb] = count_nonsolvent_residues(npz)
    print(f'  {pdb}: {counts[pdb]} non-solvent residues')

chosen_pdb = max(counts, key=counts.__getitem__)
struct = Structure.load(STRUCTURES_DIR / f'{chosen_pdb}.npz')

centers, is_water, _ = get_residue_centers(struct)
n_polymer = int((~is_water).sum())
n_water   = int(is_water.sum())

print(f'\nSelected: {chosen_pdb}')
print(f'  Cluster size: {len(cluster)}')
print(f'  Present residue centers: {len(centers)}')
print(f'  Non-water (polymer + ligand): {n_polymer}')
print(f'  Water residues: {n_water}')

  5kdo: 773 non-solvent residues
  6crk: 1032 non-solvent residues
  6rmv: 430 non-solvent residues
  8qeg: 785 non-solvent residues
  8qeh: 789 non-solvent residues

Selected: 6crk
  Cluster size: 5
  Present residue centers: 1609
  Non-water (polymer + ligand): 982
  Water residues: 627


## Section 2 — Compute crops

Run greedy sphere-covering with `radius=20.0 Å`.
Any water-only or isolated crops are printed as warnings below.
A summary table follows immediately.

In [6]:
RADIUS = 30.0  # Å

print(f'Running greedy crop (radius={RADIUS} Å) on {chosen_pdb} ...')
print('Warnings (if any):')
crops = greedy_crop(struct, radius=RADIUS)
print(f'\nTotal crops: {len(crops)}')
print()

# Summary table
hdr = f"{'Crop':>4}  {'Non-water res':>14}  {'Waters':>7}  {'Total':>6}"
print(hdr)
print('-' * len(hdr))
for i, crop in enumerate(crops):
    total = len(crop['res_global_indices'])
    print(f"{i:>4}  {crop['n_nonsolvent']:>14}  {crop['n_water']:>7}  {total:>6}")
print('-' * len(hdr))
total_ns = sum(c['n_nonsolvent'] for c in crops)
total_w  = sum(c['n_water'] for c in crops)
total_all = sum(len(c['res_global_indices']) for c in crops)
print(f"{'SUM':>4}  {total_ns:>14}  {total_w:>7}  {total_all:>6}")
print()
print(f'Structure totals: {n_polymer} non-water, {n_water} water, {n_polymer+n_water} total')
print('(Crops may overlap, so sums can exceed structure totals.)')

Running greedy crop (radius=30.0 Å) on 6crk ...
Warnings (if any):

Total crops: 12

Crop   Non-water res   Waters   Total
-------------------------------------
   0             215      140     355
   1             289      217     506
   2             351      271     622
   3             161      126     287
   4             164      108     272
   5              82       53     135
   6             199      115     314
   7             206      132     338
   8             386      265     651
   9             246      135     381
  10             103       38     141
  11             192       63     255
-------------------------------------
 SUM            2594     1663    4257

Structure totals: 982 non-water, 627 water, 1609 total
(Crops may overlap, so sums can exceed structure totals.)


## Section 3 — Save crops as mmCIF

Each crop is saved to `outputs/crops/<pdb>_crop_<nn>.cif` using BoltzGen's
`Structure.extract_residues` + `to_mmcif` pipeline, which handles all
reindexing correctly.

In [14]:
for i, crop in enumerate(crops):
    out = OUTPUT_DIR / f'{chosen_pdb}' / f'radius_{RADIUS}' f'_crop_{i:02d}.cif'
    save_crop_as_mmcif(struct, crop['res_global_indices'], out)
    print(
        f'Saved {out.name}  '
        f'({crop["n_nonsolvent"]} non-water res, {crop["n_water"]} waters)'
    )

print(f'\nAll crops saved to {OUTPUT_DIR}')

FileNotFoundError: [Errno 2] No such file or directory: '/data/rbg/users/gloriama/dev/water_conservation/outputs/crops/6crk/radius_30.0_crop_00.cif'